=============================================================================
Operational Risk Clustering
"Where are delivery incidents concentrated — by courier type, shift pattern,
 and city — and what is driving them?"
=============================================================================
Run each cell block in a Jupyter notebook.
Input: master_table.csv + cleanincidents.csv (produced from the earlier merge)
=============================================================================

## 0. Imports & Config


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Update these paths if needed
MASTER_PATH    = 'master_table.csv'
INCIDENTS_PATH = 'cleanincidents.csv'


## 1. Load Data


In [ ]:
master    = pd.read_csv(MASTER_PATH)
incidents = pd.read_csv(INCIDENTS_PATH)

print(f"Master table : {master.shape}")
print(f"Incidents    : {incidents.shape}")
master.head(2)


## 2. Build Courier-Level Risk Profile

We aggregate per named courier (excluding 'Subcontractor' rows which have no
shift / employment metadata). Each courier becomes one row with numeric risk
features that the clustering model will learn from.


In [ ]:
named = master[master['courier_id'] != 'Subcontractor'].copy()

# Core delivery metrics per courier
courier_profile = (
    named
    .groupby(['courier_id', 'employment_type', 'courier_city', 'shift_pattern'])
    .agg(
        total_deliveries    = ('delivery_id', 'count'),
        incident_deliveries = ('has_incident', 'sum'),
        failed_deliveries   = ('delivery_status', lambda x: x.isin(['Failed', 'Returned']).sum()),
        avg_time_taken      = ('time_taken_minutes', 'mean'),
    )
    .reset_index()
)

# Count each raw incident type per courier
inc_detail = (
    named[['delivery_id', 'courier_id']]
    .merge(incidents[['delivery_id', 'incident_type']], on='delivery_id', how='left')
)
inc_counts = (
    inc_detail
    .groupby('courier_id')['incident_type']
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

courier_profile = courier_profile.merge(inc_counts, on='courier_id', how='left')

# Convert raw counts to rates (per delivery)
n = courier_profile['total_deliveries']
courier_profile['incident_rate']      = courier_profile['incident_deliveries'] / n
courier_profile['failure_rate']       = courier_profile['failed_deliveries']   / n
courier_profile['rate_damaged']       = courier_profile.get('Damaged parcel',      0) / n
courier_profile['rate_late']          = courier_profile.get('Late delivery',        0) / n
courier_profile['rate_complaint']     = courier_profile.get('Customer complaint',   0) / n
courier_profile['rate_lost']          = courier_profile.get('Lost parcel',          0) / n
courier_profile['rate_breakdown']     = courier_profile.get('Vehicle breakdown',    0) / n
courier_profile['rate_wrong_address'] = courier_profile.get('Wrong address',        0) / n

print(f"Couriers in profile: {len(courier_profile)}")
courier_profile[['courier_id','employment_type','courier_city','shift_pattern',
                 'total_deliveries','incident_rate','failure_rate']].head(10)


## 3. Feature Matrix


In [ ]:
num_features = [
    'incident_rate', 'failure_rate', 'avg_time_taken',
    'rate_damaged', 'rate_late', 'rate_complaint',
    'rate_lost', 'rate_breakdown', 'rate_wrong_address'
]

cat_dummies = pd.get_dummies(
    courier_profile[['employment_type', 'courier_city', 'shift_pattern']],
    drop_first=False
)

X = pd.concat([courier_profile[num_features], cat_dummies], axis=1).fillna(0)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)


## 4. Choose k — Elbow + Silhouette


In [ ]:
ks        = range(2, 9)
inertias  = []
sil_scores = []

for k in ks:
    km  = KMeans(n_clusters=k, random_state=42, n_init=50)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(ks), inertias, marker='o', color='steelblue')
axes[0].set_title('Elbow — Inertia vs k')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(ks), sil_scores, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

print("Silhouette scores:", {k: round(s, 3) for k, s in zip(ks, sil_scores)})


## 5. Fit Final Model

k=3 gives a clean low / medium / high risk split and is interpretable with
only 65 couriers. Adjust if the elbow/silhouette charts suggest otherwise.


In [ ]:
K_FINAL = 3

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=50)
courier_profile['cluster'] = km_final.fit_predict(X_scaled)

# Rank clusters by incident rate so cluster 0 = lowest risk
rank_map = (
    courier_profile.groupby('cluster')['incident_rate']
    .mean()
    .rank()
    .sub(1)
    .astype(int)
    .to_dict()
)
courier_profile['cluster'] = courier_profile['cluster'].map(rank_map)

cluster_labels = {0: 'Low Risk', 1: 'Medium Risk', 2: 'High Risk'}
courier_profile['cluster_label'] = courier_profile['cluster'].map(cluster_labels)

courier_profile[['courier_id','employment_type','courier_city','shift_pattern',
                 'cluster','cluster_label','incident_rate']].sort_values('cluster')


## 6. Cluster Profiles


In [ ]:
profile = (
    courier_profile.groupby('cluster_label')
    .agg(
        n_couriers          = ('courier_id', 'count'),
        avg_incident_rate   = ('incident_rate', 'mean'),
        avg_failure_rate    = ('failure_rate', 'mean'),
        avg_time_taken      = ('avg_time_taken', 'mean'),
        avg_rate_damaged    = ('rate_damaged', 'mean'),
        avg_rate_late       = ('rate_late', 'mean'),
        avg_rate_complaint  = ('rate_complaint', 'mean'),
        avg_rate_lost       = ('rate_lost', 'mean'),
        avg_rate_breakdown  = ('rate_breakdown', 'mean'),
        avg_rate_wrong_addr = ('rate_wrong_address', 'mean'),
    )
    .round(4)
)
print(profile.T)


## 7. Composition Breakdown — What's in Each Cluster?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cluster Composition', fontsize=14, fontweight='bold')

colors = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 'High Risk': '#e74c3c'}

# Employment type
emp = courier_profile.groupby(['cluster_label', 'employment_type']).size().unstack(fill_value=0)
emp.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white')
axes[0].set_title('Employment Type')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# Shift pattern
shift = courier_profile.groupby(['cluster_label', 'shift_pattern']).size().unstack(fill_value=0)
shift.plot(kind='bar', ax=axes[1], colormap='Set1', edgecolor='white')
axes[1].set_title('Shift Pattern')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

# City
city = courier_profile.groupby(['cluster_label', 'courier_city']).size().unstack(fill_value=0)
city.plot(kind='bar', ax=axes[2], colormap='tab10', edgecolor='white')
axes[2].set_title('Courier City')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 8. Incident Type Breakdown by Cluster


In [ ]:
named_clustered = named.merge(
    courier_profile[['courier_id', 'cluster', 'cluster_label']], on='courier_id', how='left'
)

inc_by_cluster = (
    named_clustered[['delivery_id', 'cluster_label']]
    .merge(incidents[['delivery_id', 'incident_type']], on='delivery_id', how='inner')
    .groupby(['cluster_label', 'incident_type'])
    .size()
    .reset_index(name='count')
)

totals = named_clustered.groupby('cluster_label')['delivery_id'].count().reset_index(name='total')
inc_by_cluster = inc_by_cluster.merge(totals, on='cluster_label')
inc_by_cluster['rate'] = inc_by_cluster['count'] / inc_by_cluster['total']

pivot = inc_by_cluster.pivot(index='incident_type', columns='cluster_label', values='rate')
pivot = pivot[['Low Risk', 'Medium Risk', 'High Risk']]

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax,
           color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white', width=0.7)
ax.set_title('Incident Type Rate by Cluster (per delivery)')
ax.set_xlabel('')
ax.set_ylabel('Rate (incidents / total deliveries)')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Cluster')
plt.tight_layout()
plt.show()


## 9. City × Shift Pattern Heatmap


In [ ]:
seg = (
    named_clustered
    .groupby(['delivery_city', 'shift_pattern'])
    .agg(total=('delivery_id', 'count'), incidents=('has_incident', 'sum'))
    .reset_index()
)
seg['incident_rate'] = seg['incidents'] / seg['total']

heat = seg.pivot(index='delivery_city', columns='shift_pattern', values='incident_rate')

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heat, annot=True, fmt='.1%', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Incident Rate'}
)
ax.set_title('Incident Rate — City × Shift Pattern')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## 10. PCA Scatter — Couriers in 2D


In [ ]:
pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
courier_profile['pca_x'] = coords[:, 0]
courier_profile['pca_y'] = coords[:, 1]

print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2={pca.explained_variance_ratio_[1]:.1%}")

fig, ax = plt.subplots(figsize=(9, 6))
color_map = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}

for cluster, grp in courier_profile.groupby('cluster'):
    ax.scatter(
        grp['pca_x'], grp['pca_y'],
        c=color_map[cluster], label=cluster_labels[cluster],
        s=90, edgecolors='white', linewidth=0.8, zorder=3
    )
    for _, row in grp.iterrows():
        ax.annotate(
            row['courier_id'], (row['pca_x'], row['pca_y']),
            fontsize=6, ha='center', va='bottom', color='#333'
        )

ax.set_title('Courier Clusters — PCA projection (PC1 + PC2)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(title='Cluster')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 11. Export Results


In [ ]:
# Courier-level cluster assignments + metrics
courier_profile.to_csv('courier_clusters.csv', index=False)

# Master table enriched with cluster label
master_enriched = master.merge(
    courier_profile[['courier_id', 'cluster', 'cluster_label']],
    on='courier_id', how='left'
)
master_enriched.to_csv('master_table_clustered.csv', index=False)

print("Saved:")
print("  courier_clusters.csv        — one row per courier with cluster label + all metrics")
print("  master_table_clustered.csv  — original master with cluster column added")


# =============================================================================
# SECTION B — Feature Distinctiveness per Cluster
# Run after Section A (cells 1–5) to understand what defines each profile
# =============================================================================


## 12. Standardised Cluster Means (Z-Score Heatmap)

Shows how far each cluster sits from the global average on every feature.
Values near 0 = average. Positive = above average. Negative = below average.
This is the clearest way to see what makes each cluster distinct.


In [ ]:
feature_names = list(X.columns)
X_df = pd.DataFrame(X_scaled, columns=feature_names)
X_df['cluster'] = courier_profile['cluster'].values

z_means = X_df.groupby('cluster')[feature_names].mean()

# Readable row labels
label_map = {
    'incident_rate': 'Overall incident rate',
    'failure_rate': 'Failure / return rate',
    'avg_time_taken': 'Avg. time taken (mins)',
    'rate_damaged': '↳ Damaged parcel',
    'rate_late': '↳ Late delivery',
    'rate_complaint': '↳ Customer complaint',
    'rate_lost': '↳ Lost parcel',
    'rate_breakdown': '↳ Vehicle breakdown',
    'rate_wrong_address': '↳ Wrong address',
    'employment_type_Contracted': 'Contracted courier',
    'employment_type_Employed': 'Employed courier',
    'shift_pattern_Day': 'Day shift',
    'shift_pattern_Evening': 'Evening shift',
    'shift_pattern_Mixed': 'Mixed shift',
    'shift_pattern_Night': 'Night shift',
    'courier_city_Birmingham': 'City: Birmingham',
    'courier_city_Bristol': 'City: Bristol',
    'courier_city_Glasgow': 'City: Glasgow',
    'courier_city_Leeds': 'City: Leeds',
    'courier_city_London': 'City: London',
    'courier_city_Manchester': 'City: Manchester',
}

z_plot = z_means.T.rename(index=label_map)
z_plot.columns = ['Low Risk', 'Medium Risk', 'High Risk']

# Separate numeric rates from categoricals for visual grouping
rate_rows = [label_map[f] for f in feature_names if f.startswith('rate_') or f in ('incident_rate','failure_rate','avg_time_taken')]
cat_rows  = [label_map[f] for f in feature_names if f not in ('incident_rate','failure_rate','avg_time_taken') and not f.startswith('rate_')]

fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={'width_ratios': [1, 1]})
fig.suptitle('Feature Z-Scores by Cluster  (0 = global average)', fontsize=13, fontweight='bold')

for ax, rows, title in zip(axes, [rate_rows, cat_rows], ['Incident & Rate Features', 'Courier Characteristics']):
    subset = z_plot.loc[rows]
    sns.heatmap(
        subset, annot=True, fmt='.2f', center=0,
        cmap='RdYlGn_r', linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Z-score'},
        vmin=-2.5, vmax=2.5
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


## 13. Decision Tree — Minimal Rule Set

Trains a shallow decision tree to predict cluster membership.
The splits tell you the simplest rule that separates the clusters.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_scaled, courier_profile['cluster'])

print("Decision rules (scaled feature values):\n")
print(export_text(dt, feature_names=feature_names, max_depth=4))

fig, ax = plt.subplots(figsize=(14, 5))
plot_tree(
    dt, feature_names=feature_names,
    class_names=['Low', 'Medium', 'High'],
    filled=True, rounded=True, fontsize=8, ax=ax
)
ax.set_title('Decision Tree — Cluster Separation Rules', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 14. Random Forest Feature Importances

Uses 200 trees to measure which features most reliably separate the clusters.
Employment type and shift_pattern_Night should dominate.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_scaled, courier_profile['cluster'])

rf_imp = (
    pd.Series(rf.feature_importances_, index=feature_names)
    .rename(index=label_map)
    .sort_values(ascending=True)
)
# Only show features with meaningful importance
rf_imp = rf_imp[rf_imp > 0.005]

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#e74c3c' if v > 0.08 else '#f39c12' if v > 0.04 else '#95a5a6' for v in rf_imp.values]
rf_imp.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0.05, linestyle='--', color='#333', linewidth=0.8, label='5% threshold')
ax.set_title('Random Forest Feature Importances — Cluster Separation', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
ax.legend()
plt.tight_layout()
plt.show()

print(rf_imp.sort_values(ascending=False).round(4))


## 15. Parallel Coordinates — Each Courier as a Line

Every courier is drawn as a line across all numeric features.
Lines are coloured by cluster. Crossing lines reveal trade-offs;
parallel lines within a cluster confirm tight grouping.


In [ ]:
from pandas.plotting import parallel_coordinates

pc_features = [
    'incident_rate', 'failure_rate',
    'rate_damaged', 'rate_late', 'rate_complaint',
    'rate_lost', 'rate_breakdown', 'rate_wrong_address'
]

pc_df = courier_profile[pc_features + ['cluster_label']].copy()

# Normalise each feature to [0, 1] for legibility
for col in pc_features:
    mn, mx = pc_df[col].min(), pc_df[col].max()
    pc_df[col] = (pc_df[col] - mn) / (mx - mn + 1e-9)

fig, ax = plt.subplots(figsize=(13, 5))
parallel_coordinates(
    pc_df, 'cluster_label',
    color=['#2ecc71', '#f39c12', '#e74c3c'],
    alpha=0.55, linewidth=1.4, ax=ax
)
ax.set_title('Parallel Coordinates — Courier Risk Profiles by Cluster', fontsize=12, fontweight='bold')
ax.set_xticklabels([label_map.get(f, f) for f in pc_features], rotation=25, ha='right')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


---
## Key Findings Summary

| Cluster    | Primary Drivers                              | Distinct Features                            |
|------------|----------------------------------------------|----------------------------------------------|
| Low Risk   | Employed couriers, Day shift                 | Employed (z=+1.35), Day shift (z=+0.66), all incident rates z ≈ -1.0 |
| Medium Risk| Contracted couriers, Evening / Mixed shift   | Contracted (z=+0.68), Evening (z=+0.32), incident rates near average |
| High Risk  | Contracted couriers, Night shift             | Night shift (z=+2.1) is the single strongest signal; all incident type rates z ≈ +1.6 |

**Critical insight:** The decision tree separates all three clusters on `incident_rate` alone —
meaning employment type and shift pattern don't add independent signal once you know the
incident rate. However, night shift is the *cause* (z=+2.1), not a symptom.
Individual incident types (damaged, late, complaint, etc.) are all elevated
proportionally in the high-risk group — no single failure mode dominates —
which points to a systemic operational issue with night-shift contracted
couriers rather than a specific fixable problem (e.g. routing or packaging).


=============================================================================
SECTION C — Revenue Impact Analysis
Translates cluster risk profiles into revenue terms using revenue_on_failed_flag
=============================================================================


## 16. Revenue Setup — Merge Clusters onto Master

`revenue_on_failed_flag = True` marks every Failed or Returned delivery.
These rows have `revenue_gbp > 0` but `realised_revenue_gbp = 0` — revenue
that was expected but never collected. We use the gap between the two columns
as the revenue lost figure.


In [ ]:
master_c = master.merge(
    courier_profile[['courier_id', 'cluster', 'cluster_label']],
    on='courier_id', how='left'
)
master_c['cluster_label'] = master_c['cluster_label'].fillna('Subcontractor')
master_c['revenue_lost']  = master_c['revenue_gbp'] - master_c['realised_revenue_gbp']

# Quick sanity check
total_flagged = master_c['revenue_on_failed_flag'].sum()
total_lost    = master_c['revenue_lost'].sum()
print(f"Flagged deliveries (Failed + Returned): {total_flagged:,}")
print(f"Total revenue lost:                     £{total_lost:,.2f}")
print(f"Total gross revenue:                    £{master_c['revenue_gbp'].sum():,.2f}")
print(f"Overall revenue loss rate:              {total_lost / master_c['revenue_gbp'].sum() * 100:.2f}%")

# Date range for annualisation
master_c['delivery_date'] = pd.to_datetime(master_c['delivery_date'])
dataset_days = (master_c['delivery_date'].max() - master_c['delivery_date'].min()).days
print(f"\nDataset spans {dataset_days} days "
      f"({master_c['delivery_date'].min().date()} → {master_c['delivery_date'].max().date()})")
print(f"Annualised revenue at risk: £{total_lost / dataset_days * 365:,.2f}")


## 17. Revenue Waterfall — Gross → Realised → Lost by Cluster

Medium Risk holds the largest absolute loss purely because it carries the most
deliveries. The % loss rate tells a different, more important story.


In [ ]:
rev = (
    master_c.groupby('cluster_label')
    .agg(
        deliveries       = ('delivery_id', 'count'),
        flagged          = ('revenue_on_failed_flag', 'sum'),
        gross_revenue    = ('revenue_gbp', 'sum'),
        realised_revenue = ('realised_revenue_gbp', 'sum'),
        revenue_lost     = ('revenue_lost', 'sum'),
    )
    .reset_index()
)
rev['pct_lost']    = rev['revenue_lost']    / rev['gross_revenue'] * 100
rev['failure_rate']= rev['flagged']         / rev['deliveries']    * 100
rev['incident_rate'] = master_c.groupby('cluster_label')['has_incident'].mean().values * 100

# Order for display
order = ['Low Risk', 'Medium Risk', 'High Risk', 'Subcontractor']
rev   = rev.set_index('cluster_label').reindex(order).reset_index()

print(rev[['cluster_label','deliveries','gross_revenue','realised_revenue',
           'revenue_lost','pct_lost','failure_rate','incident_rate']].round(2).to_string(index=False))

# Waterfall chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Revenue Impact by Cluster', fontsize=13, fontweight='bold')

palette = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12',
           'High Risk': '#e74c3c', 'Subcontractor': '#95a5a6'}
colors  = [palette[c] for c in rev['cluster_label']]

# Left: absolute revenue (stacked gross vs lost)
x = np.arange(len(rev))
axes[0].bar(x, rev['realised_revenue'], color=colors, alpha=0.85, label='Realised', edgecolor='white')
axes[0].bar(x, rev['revenue_lost'], bottom=rev['realised_revenue'],
            color=colors, alpha=0.35, hatch='///', edgecolor='white', label='Lost (flagged)')
for i, row in rev.iterrows():
    axes[0].text(i, row['gross_revenue'] + 500, f"£{row['revenue_lost']:,.0f}\nlost",
                 ha='center', fontsize=8, color='#333')
axes[0].set_xticks(x)
axes[0].set_xticklabels(rev['cluster_label'], rotation=0)
axes[0].set_ylabel('Revenue (£)')
axes[0].set_title('Absolute Revenue: Realised vs Lost')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'£{v:,.0f}'))

# Right: % loss rate vs incident rate side by side
width = 0.35
axes[1].bar(x - width/2, rev['incident_rate'], width, color=colors, alpha=0.9,
            label='Incident rate %', edgecolor='white')
axes[1].bar(x + width/2, rev['pct_lost'],     width, color=colors, alpha=0.4,
            label='Revenue loss rate %', edgecolor='white', hatch='///')
for i, row in rev.iterrows():
    axes[1].text(i - width/2, row['incident_rate'] + 0.3, f"{row['incident_rate']:.1f}%",
                 ha='center', fontsize=8)
    axes[1].text(i + width/2, row['pct_lost'] + 0.3, f"{row['pct_lost']:.1f}%",
                 ha='center', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(rev['cluster_label'], rotation=0)
axes[1].set_ylabel('Rate (%)')
axes[1].set_title('Incident Rate vs Revenue Loss Rate')
axes[1].legend()

plt.tight_layout()
plt.show()


## 18. The Key Disconnect — Incident Rate ≠ Revenue Loss Rate

This chart makes the critical finding explicit: despite the High Risk cluster
having a 35% incident rate (vs 15% for Low Risk), all clusters lose revenue
at almost the same rate (~8%). Incidents are being absorbed operationally —
they create cost and customer friction but do not consistently kill the delivery.
The ~8% failure rate appears structural, not cluster-specific.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

named_only = rev[rev['cluster_label'] != 'Subcontractor'].copy()

ax.scatter(named_only['incident_rate'], named_only['pct_lost'],
           c=[palette[c] for c in named_only['cluster_label']],
           s=named_only['deliveries'] / 150,   # size = volume of deliveries
           alpha=0.85, edgecolors='white', linewidth=1.2, zorder=3)

for _, row in named_only.iterrows():
    ax.annotate(
        f"{row['cluster_label']}\n({row['deliveries']:,} deliveries)",
        (row['incident_rate'], row['pct_lost']),
        textcoords='offset points', xytext=(8, 4), fontsize=9, color='#333'
    )

ax.axhline(rev['pct_lost'].mean(), linestyle='--', color='#aaa', linewidth=1,
           label=f"Avg revenue loss rate ({rev['pct_lost'].mean():.1f}%)")
ax.set_xlabel('Incident Rate (%)', fontsize=11)
ax.set_ylabel('Revenue Loss Rate (% of gross)', fontsize=11)
ax.set_title('Incident Rate vs Revenue Loss Rate\n(bubble size = delivery volume)',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("Key finding: A 2.4× difference in incident rate produces only a 0.3pp difference")
print("in revenue loss rate. Incidents ≠ revenue loss — most incidents are resolved.")


## 19. Recoverable Revenue — Counterfactual Analysis

How much revenue would be recovered if the Medium Risk and High Risk clusters
were brought down to the Low Risk cluster's failure rate (the baseline)?


In [ ]:
baseline_failure_rate = rev.loc[rev['cluster_label']=='Low Risk', 'failure_rate'].values[0] / 100
avg_rev_per_delivery  = master_c['revenue_gbp'].mean()

recoverable = []
for _, row in rev[rev['cluster_label'].isin(['Medium Risk', 'High Risk', 'Subcontractor'])].iterrows():
    actual_lost      = row['revenue_lost']
    counterfactual   = row['deliveries'] * baseline_failure_rate * (row['gross_revenue'] / row['deliveries'])
    rec              = max(actual_lost - counterfactual, 0)
    recoverable.append({
        'cluster_label':      row['cluster_label'],
        'actual_lost':        actual_lost,
        'expected_at_baseline': counterfactual,
        'recoverable':        rec,
        'pct_of_lost':        rec / actual_lost * 100 if actual_lost > 0 else 0
    })

rec_df = pd.DataFrame(recoverable)
total_rec = rec_df['recoverable'].sum()

print("=== Counterfactual: bring all clusters to Low Risk failure rate "
      f"({baseline_failure_rate:.2%}) ===\n")
print(rec_df.round(2).to_string(index=False))
print(f"\nTotal recoverable:         £{total_rec:,.2f}")
print(f"Annualised recoverable:    £{total_rec / dataset_days * 365:,.2f}")
print(f"As % of total realised:    {total_rec / master_c['realised_revenue_gbp'].sum() * 100:.2f}%")
print(f"\nNote: The modest recoverable amount ({total_rec / master_c['realised_revenue_gbp'].sum() * 100:.2f}%)")
print("confirms the structural nature of the ~8% failure rate — it is not")
print("materially driven by courier cluster. The bigger lever is the total")
print(f"£{total_lost:,.2f} in absolute lost revenue (£{total_lost/dataset_days*365:,.2f} annualised).")


In [ ]:
# Visualise recoverable vs non-recoverable within lost revenue
fig, ax = plt.subplots(figsize=(9, 5))

all_clusters = rev[rev['cluster_label'] != 'Low Risk'].copy()
rec_dict = rec_df.set_index('cluster_label')['recoverable'].to_dict()
all_clusters['recoverable']     = all_clusters['cluster_label'].map(rec_dict).fillna(0)
all_clusters['non_recoverable'] = all_clusters['revenue_lost'] - all_clusters['recoverable']

x = np.arange(len(all_clusters))
ax.bar(x, all_clusters['non_recoverable'],
       color=[palette[c] for c in all_clusters['cluster_label']],
       alpha=0.85, label='Structural loss (baseline rate)', edgecolor='white')
ax.bar(x, all_clusters['recoverable'],
       bottom=all_clusters['non_recoverable'],
       color=[palette[c] for c in all_clusters['cluster_label']],
       alpha=0.4, hatch='///', label='Recoverable (above baseline)', edgecolor='white')

for i, row in all_clusters.iterrows():
    ax.text(i, row['revenue_lost'] + 80,
            f"£{row['recoverable']:,.0f}\nrecoverable",
            ha='center', fontsize=8, color='#555')

ax.set_xticks(x)
ax.set_xticklabels(all_clusters['cluster_label'])
ax.set_ylabel('Revenue Lost (£)')
ax.set_title('Revenue Lost: Structural vs Recoverable by Cluster', fontsize=12, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'£{v:,.0f}'))
plt.tight_layout()
plt.show()


## 20. Revenue Lost by City × Cluster (Heatmap)

Absolute revenue lost — useful for prioritising where to focus operationally.


In [ ]:
city_cluster_rev = (
    master_c[master_c['cluster_label'] != 'Subcontractor']
    .groupby(['delivery_city', 'cluster_label'])
    .agg(revenue_lost=('revenue_lost', 'sum'), deliveries=('delivery_id', 'count'))
    .reset_index()
)
city_cluster_rev['pct_lost'] = city_cluster_rev['revenue_lost'] / city_cluster_rev['deliveries'] * 100

# Two heatmaps: absolute £ and % rate
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Revenue Lost by City × Cluster', fontsize=13, fontweight='bold')

for ax, val_col, fmt, title in zip(
    axes,
    ['revenue_lost', 'pct_lost'],
    ['£{:.0f}', '{:.1f}%'],
    ['Absolute Revenue Lost (£)', 'Revenue Loss Rate (% per delivery)']
):
    pivot = city_cluster_rev.pivot(index='delivery_city', columns='cluster_label', values=val_col)
    pivot = pivot[['Low Risk', 'Medium Risk', 'High Risk']]
    labels = pivot.applymap(lambda v: fmt.format(v) if pd.notna(v) else '')
    sns.heatmap(pivot, annot=labels, fmt='', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': val_col})
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


## 21. Incident Type Co-occurrence on Flagged (Revenue-Lost) Deliveries

Every failed/returned delivery has an incident logged (100% co-occurrence).
This shows which incident types appear most on deliveries that actually lost
revenue — useful for prioritising which incident types are highest stakes.


In [ ]:
flagged_inc = (
    master_c[master_c['revenue_on_failed_flag'] == True]
    .merge(incidents[['delivery_id', 'incident_type']], on='delivery_id', how='left')
)

inc_rev = (
    flagged_inc.groupby(['cluster_label', 'incident_type'])
    .agg(count=('delivery_id','count'), revenue_at_risk=('revenue_lost','sum'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Incident Types on Revenue-Lost Deliveries', fontsize=13, fontweight='bold')

# Count of flagged deliveries per incident type per cluster
pivot_count = inc_rev.pivot(index='incident_type', columns='cluster_label', values='count').fillna(0)
for col in ['Low Risk','Medium Risk','High Risk']:
    if col not in pivot_count.columns:
        pivot_count[col] = 0
pivot_count[['Low Risk','Medium Risk','High Risk']].plot(
    kind='bar', ax=axes[0],
    color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white', width=0.7
)
axes[0].set_title('Count of Flagged Deliveries by Incident Type')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Cluster')

# Revenue at risk per incident type (summed across clusters)
rev_by_type = inc_rev.groupby('incident_type')['revenue_at_risk'].sum().sort_values(ascending=True)
colors_bar  = ['#e74c3c' if v == rev_by_type.max() else '#f39c12' if v >= rev_by_type.median() else '#95a5a6'
                for v in rev_by_type.values]
rev_by_type.plot(kind='barh', ax=axes[1], color=colors_bar, edgecolor='white')
axes[1].set_title('Total Revenue at Risk by Incident Type (all clusters)')
axes[1].set_xlabel('Revenue Lost (£)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'£{v:,.0f}'))

plt.tight_layout()
plt.show()


---
## Revenue Impact Summary

| Metric | Value |
|---|---|
| Total gross revenue (dataset) | £492,095 |
| Total realised revenue | £452,852 |
| Total revenue lost (flagged) | £39,243 |
| Overall revenue loss rate | ~8% |
| Annualised revenue at risk | ~£52,660 |
| Recoverable by fixing High+Med Risk to Low Risk baseline | ~£943 (0.2%) |

**The critical insight:** Despite the High Risk cluster having a 2.4× higher incident
rate than Low Risk (35% vs 15%), all three clusters lose revenue at effectively the
same rate (~7.8–8.1%). The ~8% failure rate is **structural** — it does not vary
meaningfully by courier type, employment status, or shift pattern.

This means:
- Reducing incidents in the High Risk cluster (night-shift contractors) will recover
  operational cost and improve customer experience, but will not materially recover revenue.
- The real revenue lever is the £39k/£53k annualised loss that sits across all clusters.
  Investigating why ~8% of all deliveries — regardless of cluster — fail or are returned
  is the higher-priority revenue question.
- **Customer complaints and late delivery** are the most frequent incident types on
  revenue-lost deliveries — these are candidates for root-cause investigation.


=============================================================================
SECTION D — Resolution Status Analysis (requires master_table_fixed.csv)
New dimension: Resolved / Escalated / Pending — adds incident lifecycle insight
=============================================================================


## 22. Load Fixed Master Table

The fixed table correctly populates `resolution_status` with three states:
**Resolved** (~80%), **Escalated** (~10%), **Pending** (~10%).
Rows with no incident have `NaN`.


In [ ]:
master    = pd.read_csv('master_table_fixed.csv')
incidents = pd.read_csv('cleanincidents.csv')

master['revenue_lost']  = master['revenue_gbp'] - master['realised_revenue_gbp']
master['delivery_date'] = pd.to_datetime(master['delivery_date'])
master['incident_date'] = pd.to_datetime(master['incident_date'])
dataset_end             = master['incident_date'].max()

print("Resolution status breakdown:")
print(master['resolution_status'].value_counts(dropna=False))
print(f"\nDataset end date (for age calculations): {dataset_end.date()}")


## 23. Resolution Status Overview


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Resolution Status — Incident Lifecycle', fontsize=13, fontweight='bold')

has_inc = master[master['has_incident'] == True].copy()

# Pie: overall split
status_counts = has_inc['resolution_status'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c','#f39c12'],
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Overall Incident Status Split')

# By incident type
inc_joined = has_inc.merge(incidents[['delivery_id','incident_type']], on='delivery_id', how='inner')
rate_by_type = pd.crosstab(inc_joined['incident_type'], inc_joined['resolution_status'], normalize='index') * 100
rate_by_type[['Resolved','Pending','Escalated']].plot(
    kind='barh', ax=axes[1],
    color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white', stacked=True
)
axes[1].set_title('Resolution Rate by Incident Type')
axes[1].set_xlabel('%')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].axvline(80, linestyle='--', color='#aaa', linewidth=0.8)

# By city
rate_by_city = pd.crosstab(has_inc['delivery_city'], has_inc['resolution_status'], normalize='index') * 100
rate_by_city[['Resolved','Pending','Escalated']].plot(
    kind='barh', ax=axes[2],
    color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white', stacked=True
)
axes[2].set_title('Resolution Rate by City')
axes[2].set_xlabel('%')
axes[2].legend(loc='lower right', fontsize=8)
axes[2].axvline(80, linestyle='--', color='#aaa', linewidth=0.8)

plt.tight_layout()
plt.show()


## 24. Pending Incident Backlog — Age Analysis

`Pending` incidents are those raised but never closed within the dataset window.
The median pending incident has been open **154 days** — over 5 months.
This is the most actionable operational finding from the resolution dimension.


In [ ]:
pending = has_inc[has_inc['resolution_status'] == 'Pending'].copy()
pending['days_open'] = (dataset_end - pending['incident_date']).dt.days

print(f"Total pending incidents: {len(pending):,}")
print(f"\nAge distribution (days open):")
print(pending['days_open'].describe().round(1))
print(f"\nPending >30 days:   {(pending['days_open'] > 30).sum():,} ({(pending['days_open'] > 30).mean()*100:.1f}%)")
print(f"Pending >60 days:   {(pending['days_open'] > 60).sum():,} ({(pending['days_open'] > 60).mean()*100:.1f}%)")
print(f"Pending >90 days:   {(pending['days_open'] > 90).sum():,} ({(pending['days_open'] > 90).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Pending Incident Backlog', fontsize=13, fontweight='bold')

# Age histogram
axes[0].hist(pending['days_open'], bins=30, color='#f39c12', edgecolor='white', alpha=0.85)
for thresh, label, col in [(30,'30d','#e74c3c'),(60,'60d','#c0392b'),(90,'90d','#922b21')]:
    axes[0].axvline(thresh, linestyle='--', color=col, linewidth=1.2, label=f'>{label}')
axes[0].set_xlabel('Days Open')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution of Pending Incidents')
axes[0].legend()

# Pending count by city + incident type
pending_inc = pending.merge(incidents[['delivery_id','incident_type']], on='delivery_id', how='inner')
pivot_pend  = pending_inc.groupby(['delivery_city','incident_type']).size().unstack(fill_value=0)
pivot_pend.plot(kind='bar', ax=axes[1], colormap='tab10', edgecolor='white', stacked=True)
axes[1].set_title('Pending Incidents by City and Type')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()


## 25. Unresolved Revenue Exposure

Deliveries that are both **Failed/Returned** AND have **Pending or Escalated** status
represent the highest-risk revenue exposure: the delivery failed AND the incident
hasn't been actioned. These are the cases most likely to generate disputes or churn.


In [ ]:
unresolved_lost = master[
    master['resolution_status'].isin(['Pending', 'Escalated']) &
    (master['revenue_on_failed_flag'] == True)
].copy()

print(f"Deliveries: Failed/Returned with Pending or Escalated incident")
print(f"Count:           {len(unresolved_lost):,}")
print(f"Revenue at risk: £{unresolved_lost['revenue_lost'].sum():,.2f}")
print()
print(unresolved_lost.groupby(['resolution_status','delivery_status'])[['revenue_lost']].sum().round(2))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Unresolved Revenue Exposure (Failed/Returned + Pending/Escalated)', fontsize=13, fontweight='bold')

# Revenue breakdown by status combo
rev_combo = unresolved_lost.groupby(['resolution_status','delivery_status'])['revenue_lost'].sum().unstack(fill_value=0)
rev_combo.plot(kind='bar', ax=axes[0], color=['#e74c3c','#c0392b'], edgecolor='white')
axes[0].set_title('Revenue at Risk by Status Combination')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'£{v:,.0f}'))
axes[0].legend(title='Delivery Status')

# By city
rev_city = unresolved_lost.groupby('delivery_city')['revenue_lost'].sum().sort_values(ascending=True)
colors_city = ['#e74c3c' if v == rev_city.max() else '#f39c12' for v in rev_city.values]
rev_city.plot(kind='barh', ax=axes[1], color=colors_city, edgecolor='white')
axes[1].set_title('Unresolved Revenue at Risk by City')
axes[1].set_xlabel('Revenue at Risk (£)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'£{v:,.0f}'))

plt.tight_layout()
plt.show()


## 26. Resolution Rates by Cluster

Resolution rates are **flat across all clusters** (~10% Escalated, ~10% Pending,
~80% Resolved). The incident management process applies uniformly regardless of
courier risk tier — neither High Risk couriers' incidents are prioritised
nor resolved more slowly.


In [ ]:
# Merge cluster labels
master_c = master.merge(
    courier_profile[['courier_id','cluster','cluster_label']], on='courier_id', how='left'
)
master_c['cluster_label'] = master_c['cluster_label'].fillna('Subcontractor')

has_inc_c = master_c[master_c['has_incident'] == True].copy()

res_by_cluster = (
    pd.crosstab(has_inc_c['cluster_label'], has_inc_c['resolution_status'], normalize='index') * 100
)
print("Resolution rates by cluster (%):")
print(res_by_cluster.round(1))

fig, ax = plt.subplots(figsize=(9, 5))
order = ['Low Risk','Medium Risk','High Risk','Subcontractor']
res_plot = res_by_cluster.reindex(order)[['Resolved','Pending','Escalated']]
res_plot.plot(kind='bar', ax=ax, color=['#2ecc71','#f39c12','#e74c3c'],
              edgecolor='white', stacked=True)
ax.set_title('Resolution Rate by Cluster\n(flat across all — incident management does not vary by risk tier)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('% of incidents')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Status')
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()


## 27. Escalation Hotspot — Wrong Address in Glasgow

Wrong Address has the highest overall escalation rate (11%). Broken down by city,
**Glasgow shows a 16.1% escalation rate for Wrong Address** — nearly double the
next-highest city. This is a specific, addressable operational signal.


In [ ]:
wrong_addr = inc_joined[inc_joined['incident_type'] == 'Wrong address'].copy()
wa_city = (
    pd.crosstab(wrong_addr['delivery_city'], wrong_addr['resolution_status'], normalize='index') * 100
)
print("Wrong Address — escalation rate by city (%):")
print(wa_city.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Escalation Hotspot — Wrong Address by City', fontsize=13, fontweight='bold')

wa_city[['Resolved','Pending','Escalated']].plot(
    kind='bar', ax=axes[0],
    color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white', stacked=True
)
axes[0].set_title('Wrong Address: Resolution Rate by City')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('%')
axes[0].legend(title='Status')

# All incident types × city: escalation rate heatmap
all_esc = (
    pd.crosstab(
        inc_joined['incident_type'], inc_joined['delivery_city'],
        values=(inc_joined['resolution_status'] == 'Escalated').astype(int),
        aggfunc='mean'
    ) * 100
)
sns.heatmap(all_esc, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Escalation Rate (%)'},
            vmin=5, vmax=20)
axes[1].set_title('Escalation Rate (%) — Incident Type × City')
axes[1].set_xlabel('')
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


---
## Resolution Status Summary

| Finding | Detail |
|---|---|
| Pending backlog | 2,241 incidents, median age **154 days** — chronic, not acute |
| Escalated | 2,242 incidents; revenue loss rate identical to Resolved (~35%) |
| Unresolved revenue exposure | **£7,989** on deliveries that failed AND haven\'t been actioned |
| Resolution rate by cluster | Flat across all tiers — incident mgmt is process-blind to risk |
| Escalation hotspot | **Wrong address in Glasgow: 16.1% escalation rate** (vs ~10% average) |
| Key insight | Resolution status does not predict revenue recovery — once a delivery fails, the revenue is gone regardless of whether the incident is closed |
